## Lab 3 &ndash; Signatures

## Done by,
*   Mishael Natth VISWANATHAN
*   Nathan BOUDRY
*   Téophile PRADALIER-LEFEBVRE
*   Aaron Rohit JEROLD




### 0) Using signatures

Our blossoming blockchain library now contains a simple [elliptic curve](https://en.wikipedia.org/wiki/Elliptic-curve_cryptography)-based signature scheme that we will use for the purpose of the labs.

In [ ]:
from JChain import *

Let's start by creating a pair of keys for Alice:

In [ ]:
(kpriv, kpub) = KeyGen()

print("Alice's private key: %s" % kpriv)  # 32 bits - too weak for actual production use!
print("Alice's public key: %s" % kpub)    # 64 bits

Alice's private key: 32276049
Alice's public key: 173025678176026218


Note that it is always possible to derive, or extract, the public key from the private one if needed.

In [ ]:
ExtractPublicKey(kpriv)

173025678176026218

She can now append to any message her signature (computed from the message and her private key):

In [ ]:
m = "Test message"
s = Sign(kpriv, m)
(m,s)

('Test message', 338476563949534767)

When Victor (or Bob, or anyone with access to the public key) receives the signed message, he can check that the signature is valid:

In [ ]:
Verify(kpub,m,s)

True

Remark that any signature now properly computed with the `Sign` algorithm from the message and private key will (with high probability) be found invalid. This means that, in practice, only Alice (in possession of the private key) can sign messages in her name.

In [ ]:
Verify(kpub,m,42)

False

### 1) Signature support for operations

Modify the code of the generic `Operation` class so that all operations now support containing a `signature` in addition to their content. For easy refernce, the `public_key` corresponding to the signing `private_key` will be stored as well. Some methods need to be implemented:

- `Operation.sign(self, priv_key)` that computes the signature of a canonical string representation of content of the operation with `priv_key`, that will be stored in the `signature` member along with the corresponding public key;

- `Operation.is_signature_valid(self)` that checks whether the stored signature is valid for the stored `public_key`.

Provide some examples of use below.

In [ ]:
#keys generation for Alice
(kpriv, kpub) = KeyGen()

#op creation
op = Transfer({"op_type": "Transfer", "source": "Alice", "target": "Bob", "amount": 10})

#before the sign
print("Signature valid before signature? ", op.is_signature_valid()) # False

#Alice signs the operation
op.sign(kpriv)
print("Signature :", op.signature)
print("PublicKey :", op.public_key)
print("Signature valid after signature? ", op.is_signature_valid()) # True

#if someone alter
op.amount = 1000
print("Valid after alteration? ", op.is_signature_valid()) # False

Signature valid before signature?  False
Signature : 324006436447693374
PublicKey : 396111588053961442
Signature valid after signature?  True
Valid after alteration?  False


### 2) Authorized keys

Storing only signed transactions in our ledger is good, but not that much if anyone can sign anything! This means that we must also check that every operation is signed by an *authorized key* that depends on the actual contents of the operation.

Note that the `Operation.is_valid` method was modified accordingly to now check 3 things:

1. whether the stored signature is valid for the stored public key (see above);
2. whether the public key used for signing was authorized to so so;
3. whether the operation logic is valid (like we did before).

The second one needs to be implemented for each type of operation we have, and will likely depend on its content and the current state of the ledger.

Please implement this for the `Transfer` operation: the only authorized key should be the `source` account (that we now identify by the public key itself instead of the name of the holder) - and provide some examples of valid and invalid Transfers.

In [ ]:
#Create 2 users
alice_priv, alice_pub = KeyGen()
bob_priv, bob_pub = KeyGen()

# We create a transfer where Alice is the source
op = Transfer({
    "op_type": "Transfer",
    "source": alice_pub,
    "target": bob_pub,
    "amount": 10
})

# Alice sign
op.sign(alice_priv)

#op.public_key is the public key of Alice, et op.source is also the public key of Alice.
print(op.is_signature_valid())
print(op.is_key_authorized(None))

# We create a transfer where Alice is the source
op = Transfer({
    "op_type": "Transfer",
    "source": alice_pub,
    "target": bob_pub,
    "amount": 10
})

# Bob sign
op.sign(bob_priv)

print(op.is_signature_valid())
print(op.is_key_authorized(None))


True
True
True
False


`Init` operations are a bit different: only administrators of the chain should be able to use them. But how to know if a public key belongs to an administrator? We need to write that in the state of the chain... which means we need a new operation to do so. So please add a new operation type `EnrolAdminKey` that adds a given public key to the list of administrator keys of the chain (stored in the state). Then it will be possible to implement `is_key_authorized` for `Init` as well as `EnrolAdminKey`s (only admin should be authorized to add admin keys!).

In [ ]:
admin1_priv, admin1_pub = KeyGen()
admin2_priv, admin2_pub = KeyGen()
eve_priv, eve_pub = KeyGen()
alice_priv, alice_pub = KeyGen()

state = State({"admin_keys": [admin1_pub]})
print("Initial state:", state)

# 1. Admin 1 enrols Admin 2 (Valid EnrolAdminKey)
op_enrol = EnrolAdminKey({"op_type": "EnrolAdminKey", "key": admin2_pub})
op_enrol.sign(admin1_priv)
print("\n--- 1. Valid EnrolAdminKey ---")
print("Signature valid? ", op_enrol.is_signature_valid())
print("Key authorized? ", op_enrol.is_key_authorized(state))
print("Operation valid? ", op_enrol.is_valid(state))
state = op_enrol.apply(state)
print("State after enrolling admin2:", state)

# 2. Eve (non-admin) tries to enrol herself as admin (Unauthorized EnrolAdminKey)
op_eve_enrol = EnrolAdminKey({"op_type": "EnrolAdminKey", "key": eve_pub})
op_eve_enrol.sign(eve_priv)
print("\n--- 2. Unauthorized EnrolAdminKey (Eve tries to become admin) ---")
print("Key authorized? ", op_eve_enrol.is_key_authorized(state))
print("Operation valid? ", op_eve_enrol.is_valid(state))

# 3. Admin 1 tries to re-enrol Admin 2 who is already an admin (Logic Invalid)
op_re_enrol = EnrolAdminKey({"op_type": "EnrolAdminKey", "key": admin2_pub})
op_re_enrol.sign(admin1_priv)
print("\n--- 3. Invalid EnrolAdminKey (Already enroll    ed) ---")
print("Logic valid? ", op_re_enrol.is_logic_valid(state))
print("Operation valid? ", op_re_enrol.is_valid(state))

# 4. Newly enrolled Admin 2 initializes Alice's account with 100 tokens (Valid Init)
op_init = Init({"op_type": "Init", "name": alice_pub, "value": 100})
op_init.sign(admin2_priv)
print("\n--- 4. Valid Init by newly enrolled Admin 2 ---")
print("Key authorized? ", op_init.is_key_authorized(state))
print("Operation valid? ", op_init.is_valid(state))
state = op_init.apply(state)
print("State after Alice account init:", state)

# 5. Non-admin Eve tries to initialize her own balance (Unauthorized Init)
op_eve_init = Init({"op_type": "Init", "name": eve_pub, "value": 1000})
op_eve_init.sign(eve_priv)
print("\n--- 5. Unauthorized Init by Eve ---")
print("Key authorized? ", op_eve_init.is_key_authorized(state))
print("Operation valid? ", op_eve_init.is_valid(state))

Initial state: {'admin_keys': [360550749269939001]}

--- 1. Valid EnrolAdminKey ---
Signature valid?  True
Key authorized?  True
Operation valid?  True
State after enrolling admin2: {'admin_keys': [360550749269939001, 404194158224271278]}

--- 2. Unauthorized EnrolAdminKey (Eve tries to become admin) ---
Key authorized?  False
Operation valid?  False

--- 3. Invalid EnrolAdminKey (Already enrolled) ---
Logic valid?  False
Operation valid?  False

--- 4. Valid Init by newly enrolled Admin 2 ---
Key authorized?  True
Operation valid?  True
State after Alice account init: {'admin_keys': [360550749269939001, 404194158224271278], 383229173009242284: 100}

--- 5. Unauthorized Init by Eve ---
Key authorized?  False
Operation valid?  False


### 3) Valid chains of blocks

We can now put everything together and now provide ledgers of transactions that are all logically valid and signed by authorized keys. But how do we enrol the first admin key? In order for the ledger to start on an initial empty state, we need to treat the very first operation, called the `genesis` operation, differently than the others in order to set up an initial state. One way to achieve this is with yet a new kind of operation: `Genesis`, that is only (logically) valid on an empty state, and that sets up initial variables (for instance admin keys) in the state.

In [ ]:
admin1_priv, admin1_pub = KeyGen()

# 1. Create a Genesis operation setting up the initial admin key
op_genesis = Genesis({"op_type": "Genesis", "state": {"admin_keys": [admin1_pub]}})

empty_state = State({})
print("Genesis on empty state:")
print("  Logic valid?     ", op_genesis.is_logic_valid(empty_state))
print("  Key authorized?  ", op_genesis.is_key_authorized(empty_state))
print("  Signature valid? ", op_genesis.is_signature_valid())
print("  Operation valid? ", op_genesis.is_valid(empty_state))

# Apply Genesis to empty state
state = op_genesis.apply(empty_state)
print("\nState after Genesis:", state)

# 2. Genesis attempted on a non-empty state must fail
eve_priv, eve_pub = KeyGen()
bad_genesis = Genesis({"op_type": "Genesis", "state": {"admin_keys": [eve_pub]}})
print("\nGenesis on non-empty state:")
print("  Logic valid?     ", bad_genesis.is_logic_valid(state))
print("  Operation valid? ", bad_genesis.is_valid(state))

Genesis on empty state:
  Logic valid?      True
  Key authorized?   True
  Signature valid?  True
  Operation valid?  True

State after Genesis: {'admin_keys': [352397629966879831]}

Genesis on non-empty state:
  Logic valid?      False
  Operation valid?  False


With all this you should to define a `Blockchain` object, containing a list of blocks; each block containing an ordered list of valid operations as well as the hash of the previous block; and this `Blockchain` can be checked for validity (hashes + validity of all operations). Provide an example!

In [ ]:
# 1. Setup participants
admin1_priv, admin1_pub = KeyGen()
admin2_priv, admin2_pub = KeyGen()
alice_priv, alice_pub = KeyGen()
bob_priv, bob_pub = KeyGen()
eve_priv, eve_pub = KeyGen()

# 2. Prepare operations for the blocks
# Block 0: Genesis block establishing initial admin1
op0 = Genesis({"op_type": "Genesis", "state": {"admin_keys": [admin1_pub]}})

# Block 1: Admin1 enrols Admin2
op1 = EnrolAdminKey({"op_type": "EnrolAdminKey", "key": admin2_pub})
op1.sign(admin1_priv)

# Block 2: Admin2 initializes Alice with 100 tokens
op2 = Init({"op_type": "Init", "name": alice_pub, "value": 100})
op2.sign(admin2_priv)

# Block 3: Alice transfers 30 tokens to Bob
op3 = Transfer({"op_type": "Transfer", "source": alice_pub, "target": bob_pub, "amount": 30})
op3.sign(alice_priv)

# 3. Build the Blockchain
chain = Blockchain()
b0 = chain.add_block([op0])
b1 = chain.add_block([op1])
b2 = chain.add_block([op2])
b3 = chain.add_block([op3])

print(f"Blockchain created with {len(chain)} blocks.")
for i, block in enumerate(chain):
    print(f"  Block {i}: hash={block.hash()}, prev_hashes={block.prev_hashes}")

# 4. Check Blockchain validity
print("\nIs blockchain valid? ", chain.is_valid())

final_state = chain.apply_ops()
print("Final state of the blockchain:", final_state)

# 5. Tampering tests
print("\n--- Integrity & Tampering Tests ---")

# Test A: Tampering with a transaction's content (e.g., Eve alters transfer amount to 90)
tampered_chain = Blockchain([
    chain[0],
    chain[1],
    chain[2],
    Block([Transfer({"op_type": "Transfer", "source": alice_pub, "target": bob_pub, "amount": 90})], chain[3].prev_hashes)
])
print("Tampered block content valid? ", tampered_chain.is_valid())

# Test B: Tampering with block hash link
bad_link_chain = Blockchain([
    chain[0],
    Block(chain[1].content, [999999]),
    chain[2],
    chain[3]
])
print("Tampered hash link valid?     ", bad_link_chain.is_valid())

# Test C: Unauthorized transaction (Eve tries to steal Alice's tokens)
bad_op = Transfer({"op_type": "Transfer", "source": alice_pub, "target": eve_pub, "amount": 20})
bad_op.sign(eve_priv) # Signed by Eve instead of Alice
bad_chain = Blockchain([
    chain[0], chain[1], chain[2],
    Block([bad_op], [chain[2].hash()])
])
print("Unauthorized transfer valid?   ", bad_chain.is_valid())

Blockchain created with 4 blocks.
  Block 0: hash=2494508462, prev_hashes=[]
  Block 1: hash=3860344948, prev_hashes=[2494508462]
  Block 2: hash=152169297, prev_hashes=[3860344948]
  Block 3: hash=2264026328, prev_hashes=[152169297]

Is blockchain valid?  True
Final state of the blockchain: {'admin_keys': [131200032459412545, 155870676793534450], 366645900529827465: 70, 88750766155862740: 30}

--- Integrity & Tampering Tests ---
Tampered block content valid?  False
Tampered hash link valid?      False
Unauthorized transfer valid?    False
